# Entity linking: recorrido por el pipeline

Cada sección ejecuta una fase sobre datos reales. La justificación, las alternativas descartadas y las cifras completas están en `docs/memoria_resumen.md`.

Aquí se usan los primeros `LIMIT` documentos de AIDA test para que sea rápido. Las cifras de la memoria salen de `entity_linker.evaluation.evaluate` (con `document_limit=None` para el AIDA test completo) y de los scripts de `scripts/`, así que pueden diferir de las de este notebook. `RUN_LLM` activa el LLM local (Ollama).

In [ ]:
%load_ext autoreload
%autoreload 2

from html import escape

from IPython.display import HTML, display

from entity_linker.config import CONFIDENT_PROB, WEIGHTS
from entity_linker.evaluation.datasets import load_aida, load_disambiguation_sample
from entity_linker.evaluation.disambiguation import compare_disambiguation
from entity_linker.evaluation.linking import evaluate_linking
from entity_linker.evaluation.ner import evaluate_ner
from entity_linker.evaluation.retrieval import evaluate_retrieval
from entity_linker.pipeline import EntityLinkingPipeline

LIMIT = 10  # documentos de AIDA test
RUN_LLM = False  # True: el LLM local resuelve las menciones inciertas

aida = load_aida("test")
docs = aida[:LIMIT]
pipeline = EntityLinkingPipeline(use_llm=RUN_LLM)  # un solo GLiNER para todo el notebook
print(f"AIDA test: {len(aida)} documentos, se usan {len(docs)}")

In [ ]:
COLORS = {"PER": "#ffd6a5", "ORG": "#a0c4ff", "LOC": "#caffbf", "MISC": "#ffc6ff"}


def highlight(text, entities):
    """Texto con las entidades resaltadas, un color por etiqueta."""
    out, pos = [], 0
    for e in sorted(entities, key=lambda e: e.start_char):
        out.append(escape(text[pos : e.start_char]))
        out.append(
            f'<mark style="background:{COLORS.get(e.label, "#ddd")};color:#000">'
            f"{escape(e.text)} <sub>{e.label}</sub></mark>"
        )
        pos = e.end_char
    out.append(escape(text[pos:]))
    return HTML(f'<div style="white-space:pre-wrap;line-height:1.8">{"".join(out)}</div>')


def compare_f1(results):
    """{alternativa: métricas} -> tabla de F1 lado a lado."""
    names = list(results)
    print(f"{'F1':12}" + "".join(f"{n:>14}" for n in names))
    for k in results[names[0]]:
        print(f"{k:12}" + "".join(f"{results[n][k]['f1']:14.3f}" for n in names))


def show_linked(text, entities):
    """Entidades resaltadas y, por cada una, QID, confianza, revisión y reasoning."""
    display(highlight(text, entities))
    for e in entities:
        review = "REVISAR" if e.needs_review else ""
        print(
            f"{e.text[:28]:29} {e.qid:10} {(e.wikidata_label or '')[:28]:29} "
            f"{e.confidence:.2f} {review:7} | {e.reasoning}"
        )

## 1. NER: tipos finos frente a etiquetas genéricas

Misma GLiNER, dos conjuntos de etiquetas pedidas. Los tipos finos se agrupan después en PER/ORG/LOC/MISC y además alimentan la señal `type`.

In [ ]:
GENERIC_LABELS = {
    "person": "PER",
    "organization": "ORG",
    "location": "LOC",
    "miscellaneous entity": "MISC",
}
compare_f1(
    {
        "tipos finos": evaluate_ner(pipeline.ner, docs)["metrics"],
        "genéricas": evaluate_ner(pipeline.ner, docs, GENERIC_LABELS)["metrics"],
    }
)

## 2. Retrieval: el cuello de botella

Primero, qué aportan los dos refuerzos de `wbsearchentities`: la redirección de Wikipedia ("Asian Cup") y la búsqueda aproximada cuando no hay resultados ("S.Philpott").

In [ ]:
search = pipeline.wikidata._search  # acceso interno: la búsqueda sin refuerzos
for mention in ("Asian Cup", "S.Philpott"):
    plain = await search._search(mention, "en", 20)
    final = await pipeline.wikidata.get_candidates(mention)
    print(mention)
    print("  solo wbsearchentities:", [c.label for c in plain[:3]])
    print("  con refuerzos:        ", [c.label for c in final[:3]])

Recall de las menciones anotadas (`recall@all` = techo del desambiguador con 20 candidatos) y algunas menciones cuyo QID correcto no se recupera.

In [ ]:
retrieval = await evaluate_retrieval(docs, pipeline.wikidata, limit=20)
print(", ".join(f"{k} {v:.3f}" for k, v in retrieval["metrics"].items()))
print(f"\nNo recuperadas: {len(retrieval['errors'])}. Ejemplos:")
for error in retrieval["errors"][:8]:
    print(f"  {error['mention']:22} correcto {error['gold']:10} top-5 {error['candidates']}")

## 3. Desambiguación

Logit condicional sobre cinco señales, con pesos aprendidos en AIDA validation (`scripts/tune_weights.py`). La mención se acepta sin más señales si $p \geq$ `CONFIDENT_PROB`.

In [ ]:
print("Pesos:", WEIGHTS, "| CONFIDENT_PROB:", CONFIDENT_PROB)

Muestra de AIDA test que aísla la desambiguación (GLiNER acierta los límites y el correcto está entre los 20 candidatos). Se compara con el primer candidato de la búsqueda, sobre las mismas menciones (*bootstrap* pareado por documentos). Tramos: posición del correcto (`rank 1`, `rank 2-10`, `>10`) y `confident`/`uncertain` (acierto de lo que se acepta y de lo que va a revisión).

In [ ]:
test = load_disambiguation_sample("test")
report = await compare_disambiguation(test, pipeline.disambiguator, use_llm=RUN_LLM)
for name, r in report.items():
    if "diff" in r:
        print(f"{name:26} diferencia {r['diff']:+.3f}, IC 95 % {r['ci95']}")
    else:
        print(f"{name:26} {r['accuracy']}, IC 95 % {r['ci95']}")

Dos casos para leer `reasoning`: `relation` resuelve un homónimo ("Columbia" junto a Sinatra, Dylan y Santana), y el límite de la carga diferida (si las señales baratas superan `CONFIDENT_PROB`, `relation` no se calcula; memoria, sección 3.4).

In [ ]:
for text in (
    "Despite featuring some of the most prominent musicians of their decade, like "
    "Sinatra, Dylan, Joel, and Santana, Columbia was acquired by Sony in the 1980s.",
    "Michael Jordan is a professor at the University of California, Berkeley.",
):
    show_linked(text, await pipeline.link(text, use_llm=RUN_LLM))

## 4. End-to-end

NER + retrieval + desambiguación frente al gold de AIDA, con `nervaluate` y el QID como etiqueta: `mention` (límites), `all` (límites + QID), `linked` (solo con QID), `nil`.

In [ ]:
linking = await evaluate_linking(docs, pipeline)
compare_f1({"pipeline": linking["metrics"]})
print(f"\nLlamadas al LLM: {linking['llm_calls']}")

## 5. Texto libre

Sin gold: se revisa a ojo. En español, Wikidata y Wikipedia se consultan con `language="es"`.

In [ ]:
en_text = (
    "Ada Lovelace was an English mathematician and writer, chiefly known for her work "
    "on Charles Babbage's proposed mechanical general-purpose computer, the Analytical "
    "Engine. Born in London as the only legitimate child of the poet Lord Byron, she was "
    "educated privately by tutors including Augustus De Morgan of University College "
    "London. The British Computer Society awards the Lovelace Medal in her honour."
)
show_linked(en_text, await pipeline.link(en_text, use_llm=RUN_LLM))

In [ ]:
es_text = (
    "Santiago Ramón y Cajal fue un médico español especializado en histología. Nació en "
    "Petilla de Aragón, en Navarra, y estudió Medicina en la Universidad de Zaragoza. En "
    "1906 recibió el Premio Nobel de Fisiología o Medicina junto al italiano Camillo Golgi. "
    "Dirigió el Instituto Cajal, que hoy forma parte del Consejo Superior de "
    "Investigaciones Científicas."
)
show_linked(es_text, await pipeline.link(es_text, language="es", use_llm=RUN_LLM))